# <strong> 1. Prerequisites</strong>

## <Strong><font>1.1. Imports</font></Strong>: Python packages needed

In [ ]:
# Various packages
import os
import sys
import numpy as np
import pandas as pd
from pandas import DataFrame
import matplotlib.pyplot as plt
np.random.seed(42) 

# scipy
from scipy.integrate import quad_vec  # quad_vec allows to compute integrals accurately
from scipy.stats import norm
from scipy.stats import qmc 

try:
    import lets_be_rational.LetsBeRational as lbr
    print("Loaded via package import")

except ModuleNotFoundError:
    so_path = Path.home() / "Kandidatarbete/venv/lib/python3.13/site-packages/lets_be_rational/_LetsBeRational.cpython-313-darwin.so"

    if not so_path.exists():
        raise FileNotFoundError(f"Still can't find it! I am looking exactly here: {so_path}")

    # Load the C-extension directly
    spec = importlib.util.spec_from_file_location("_LetsBeRational", so_path)
    lbr = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(lbr)

    print("Loaded via direct .so import")
    print("Final module:", lbr)


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Loaded via package import


# <strong> 2. European Call & Put prices </strong>

## <Strong><font>2.1. Implement the characteristic function</font></Strong>

In [4]:
def beta_function(u, tau, sigma, rho, kappa):
    return kappa - 1j * u * sigma * rho

def alpha_hat_function(u):
    return -0.5 * u * (u + 1j)

def d_function(u, tau, sigma, rho, kappa):
    gamma = 0.5 * sigma**2
    beta = beta_function(u, tau, sigma, rho, kappa)
    alpha_hat = alpha_hat_function(u)
    return np.sqrt(beta**2 - 4 * alpha_hat * gamma)

def g_function(u, tau, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    return (beta - d) / (beta + d)

def A_function(u, tau, theta, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    common_term = np.exp(-d*tau)
    A_u = (kappa * theta / (sigma**2)) * ((beta-d)*tau - 2*np.log((g*common_term-1) / (g-1)))    
    return A_u

def B_function(u, tau, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    common_term = np.exp(-d*tau)
    B_u = ((beta-d) / (sigma**2)) * ((1 - common_term) / (1 - g*common_term))
    return B_u


In [5]:
def heston_charact_funct(u, tau, theta, sigma, rho, kappa, v0):

    beta = beta_function(u, tau, sigma, rho, kappa)    
    #alpha_hat = alpha_hat_function(u)    
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    common_term = np.exp(-d*tau)
    A = A_function(u, tau, theta, sigma, rho, kappa)
    B = B_function(u, tau, sigma, rho, kappa)

    return np.exp(A + B * v0)

## <Strong><font>2.2. Perform numerical integration</font></Strong>: using scipy integrate quad_vec function

In [6]:
def integral_price(m, tau, theta, sigma, rho, kappa, v0):
    integrand = (lambda u: 
        np.real(np.exp((1j*u + 0.5)*m)*heston_charact_funct(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return integ

## <Strong><font>2.3. Calculate European Call price </font></Strong>

In [7]:
def call_price(k, tau, S0, r, theta, sigma, rho, kappa, v0):
    m = np.log(S0/k) + r*tau #log-moneyness forward
    integ = integral_price(m, tau, theta, sigma, rho, kappa, v0)  
    price = S0 - k * np.exp(-r*tau) * integ  /np.pi
         
    return price

## <Strong><font>2.4. Calculate European put price </font></Strong>: using the call-put parity relation


In [8]:
def put_price(k, tau, S0, r, theta, sigma, rho, kappa, v0):
    price = call_price(k, tau, S0, r, theta, sigma, rho, kappa, v0)- S0 + k * np.exp(-r*tau)
    return price

## <Strong><font>2.5. Calculate the Normalised Forward Put Price $\hat{P}$ </font></Strong>

$\hat{P}(t,S_t,v_t) = \frac{e^{rT}}{K} P(t,S_t,v_t)$

In [9]:
def norm_forw_put_price(lm, r, tau, theta, sigma, rho, kappa, v0):
    m = lm + r*tau #log-moneyness forward
    integ = integral_price(m, tau, theta, sigma, rho, kappa, v0)
    return 1 - (1/np.pi) * integ

## <Strong><font>2.6. Black-Scholes as a particular case </font></Strong>
The Black-Scholes model is embedded inside the Heston model. Therefore, the European option Heston price under specific parameter values will be the Black-Scholes price. To check our Heston prices implementation, we will price European calls and put options under Black-Scholes and the Heston model with the parameter values leading to the Black-Scholes prices. We have to obtain comparable prices. Note that we cannot substitute $σ = 0$ into the Heston pricing functions because that will lead to division by zero. We will take small value of $\sigma$ (e.g $10^{-05}$).

In [11]:
def d1(k, tau, S0, r, sigma):
    return(np.log(S0/k)+(r+((sigma**2)/2.))*tau)/(sigma*np.sqrt(tau))

def d2(k, tau, S0, r, sigma):
    return d1(k,tau, S0, r, sigma) - sigma*np.sqrt(tau)

# Black-Scholes Call price 
def call_bs(k, tau, S0, r, sigma):
    return S0*norm.cdf(d1(k, tau, S0, r, sigma))-k*np.exp(-r*tau)*norm.cdf(d2(k, tau, S0, r, sigma))

# Black-Scholes Put price using the call-put parity relation
def put_bs(k, tau, S0, r, sigma):
    return call_bs(k,tau, S0, r, sigma) + k*np.exp(-r*tau) - S0    


# <strong> 3. Differential of the Normalised Forward Put Price w.r.t inputs </strong> 

In [16]:
def integral_delta(m, tau, theta, sigma, rho, kappa, v0):
    integrand = (lambda u: 
        np.real((1j*u + 0.5) * np.exp((1j*u + 0.5)*m)*heston_charact_funct(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return integ

In [17]:
def term_interm_theta(u, tau, sigma, rho, theta, kappa, v0):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)

    term1 = kappa*theta * (beta-d+2*((d*g*np.exp(-d*tau)) / ((g*np.exp(-d*tau))-1))) /(sigma**2)
    term2 = v0*(beta-d)*d * np.exp(-d*tau)*(1-g)/((sigma**2)*(1-g*np.exp(-d*tau))**2)
    term = term1 + term2

    return term    

In [18]:
def integral_differential_phi_tau(m, tau, sigma, rho, theta, kappa, v0):

    integrand = (lambda u: 
        np.real(np.exp((1j*u + 0.5)*m)*heston_charact_funct(u - 0.5j, tau, theta, sigma, rho, kappa, v0) * 
                term_interm_theta(u - 0.5j, tau, sigma, rho, theta, kappa, v0))  /(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return integ

## <Strong><font>3.1.  Differential of p_hat wrt. theta </font></Strong>

In [19]:
def differential_wrt_theta(lm, r, tau, theta, sigma, rho, kappa, v0):
    m = lm + r * tau     #log-moneyness forward
    if theta == 0.:
        theta = 0.00000001
    
    integrand = (lambda u: 
        np.real((1/theta) * np.exp((1j*u + 0.5)*m)* A_function(u - 0.5j, tau, theta, sigma, rho, kappa) * heston_charact_funct(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return (-1/np.pi) * integ

## <Strong><font>3.2. Differential of p_hat wrt. lm</font></Strong>

In [21]:
def differential_wrt_lm(lm, r, tau, theta, sigma, rho, kappa, v0):
    m = lm + r * tau #log-moneyness forward
    res = - (1/np.pi) * integral_delta(m, tau, theta, sigma, rho, kappa, v0)
    return res

## <Strong><font>3.3.  Differential of p_hat wrt. V0 </font></Strong>

In [23]:
def differential_wrt_v0(lm, r, tau, theta, sigma, rho, kappa, v0):    
    m = lm + r * tau #log-moneyness forward
    integrand = (lambda u: 
        np.real(np.exp((1j*u + 0.5)*m)* B_function(u - 0.5j, tau, sigma, rho, kappa) * heston_charact_funct(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return (-1/np.pi) * integ

## <Strong><font>3.4.  Differential of p_hat wrt. tau </font></Strong>

In [25]:
def differential_wrt_tau(lm, r, tau, theta, sigma, rho, kappa, v0):    
    m = lm + r * tau #log-moneyness forward
    integ1 = integral_delta(m, tau, theta, sigma, rho, kappa, v0)
    integ2 = integral_differential_phi_tau(m, tau, sigma, rho, theta, kappa, v0)   

    return (-1/np.pi) * (r * integ1 + integ2)

## <Strong><font>3.5.  Differential of p_hat wrt. kappa </font></Strong>

In [27]:
def differential_A_kappa(u, tau, theta, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    A_u = A_function(u, tau, theta, sigma, rho, kappa)

    res = A_u/kappa - (kappa * theta / (d * (sigma**2))) * (-d*tau+tau*beta+4*g/(g-1)+2*g*np.exp(-d*tau)*(2+tau*beta)/(1-g*np.exp(-d*tau)))
    return res

In [28]:
def differential_B_kappa(u, tau, theta, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    B_u = B_function(u, tau, sigma, rho, kappa) 

    res = -B_u/d + (1/d) * (tau*beta*np.exp(-d*tau)*B_u/(1-np.exp(-d*tau)) - g*np.exp(-d*tau)*(2+tau*beta)*B_u/(1-g*np.exp(-d*tau)))
    return res

In [29]:
def differential_phi_kappa(u, tau, theta, sigma, rho, kappa, v0):
    return heston_charact_funct(u, tau, theta, sigma, rho, kappa, v0) * (differential_A_kappa(u, tau, theta, sigma, rho, kappa) +
                                                                         v0 * differential_B_kappa(u, tau, theta, sigma, rho, kappa))


In [30]:
def differential_wrt_kappa(lm, r, tau, theta, sigma, rho, kappa, v0): 
    m = lm + r*tau   #log-moneyness forward   
    integrand = (lambda u: 
        np.real(np.exp((1j*u + 0.5)*m) * differential_phi_kappa(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return (-1/np.pi) * integ   

## <Strong><font>3.6.  Differential of p_hat wrt. rho </font></Strong>

In [32]:
def differential_A_rho(u, tau, theta, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)

    res = (kappa*theta*1j*u/(d*sigma)) * (tau*(beta-d)-2*g*(np.exp(-d*tau)*(2+tau*beta)/(g*np.exp(-d*tau)-1)-2/(g-1)))
    return res 

In [33]:
def differential_B_rho(u, tau, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    B_u = B_function(u, tau, sigma, rho, kappa) 

    res = (1j*u*sigma*B_u/d) * (1+np.exp(-d*tau)*(-tau*beta/(1-np.exp(-d*tau))+g*(2+tau*beta)/(1-g*np.exp(-d*tau))))
    return res

In [34]:
def differential_phi_rho(u, tau, theta, sigma, rho, kappa, v0):
    return heston_charact_funct(u, tau, theta, sigma, rho, kappa, v0) * (differential_A_rho(u, tau, theta, sigma, rho, kappa) +
                                                                         v0 * differential_B_rho(u, tau, sigma, rho, kappa))

In [35]:
def differential_wrt_rho(lm, r, tau, theta, sigma, rho, kappa, v0):
    m = lm + r*tau  #log-moneyness forward
    integrand = (lambda u: 
        np.real(np.exp((1j*u + 0.5)*m) * differential_phi_rho(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return (-1/np.pi) * integ   

## <Strong><font>3.7. Differential of p_hat wrt. sigma </font></Strong>

In [37]:
def differential_ln_sigma(u, tau, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    alpha_hat = alpha_hat_function(u)    

    res= ((2j*u*rho*((beta**2)-(d**2))+4*beta*alpha_hat*sigma)*((g-1)*np.exp(-d*tau)-g*np.exp(-d*tau)+1)+
          (g-1)*np.exp(-d*tau)*tau*g*((beta+d)**2)*(1j*u*rho*beta+2*alpha_hat*sigma))/(d*((beta+d)**2)*(g-1)*(g*np.exp(-d*tau)-1))
    return res

In [38]:
def differential_A_sigma(u, tau, theta, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    A_u = A_function(u, tau, theta, sigma, rho, kappa)
    diff_ln_sigma = differential_ln_sigma(u, tau, sigma, rho, kappa)
    alpha_hat = alpha_hat_function(u)

    res = -2*A_u/sigma + kappa*theta/(sigma**2)*(-1j*u*tau*rho+tau*(1j*u*rho*beta+2*alpha_hat*sigma)/d - 2*diff_ln_sigma)
    return res 

In [39]:
def differential_quotient_sigma(u, tau, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    alpha_hat = alpha_hat_function(u)

    res = ((np.exp(-d*tau)*(1j*u*rho*beta+2*alpha_hat*sigma))*(-tau*((beta+d)**2)*(1-g*np.exp(-d*tau))+(1-np.exp(-d*tau))*(2*beta+tau*g*((beta+d)**2)))
     - 2j*u*rho*(d**2)*np.exp(-d*tau)*(1-np.exp(-d*tau))) / (d*((beta+d)**2)*((1-g*np.exp(-d*tau))**2))
    return res

In [40]:
def differential_B_sigma(u, tau, sigma, rho, kappa):
    beta = beta_function(u, tau, sigma, rho, kappa)
    d = d_function(u, tau, sigma, rho, kappa)
    g = g_function(u, tau, sigma, rho, kappa)
    alpha_hat = alpha_hat_function(u)
    
    diff_quot_sigma = differential_quotient_sigma(u, tau, sigma, rho, kappa)
    res = ((1-np.exp(-d*tau))*(sigma*(-1j*u*rho*d+1j*u*rho*beta+2*alpha_hat*sigma)-2*d*(beta-d))) / (d*(sigma**3)*(1-g*np.exp(-d*tau))) + (beta-d)/(sigma**2)*diff_quot_sigma
    return res

In [41]:
def differential_phi_sigma(u, tau, theta, sigma, rho, kappa, v0):
    return heston_charact_funct(u, tau, theta, sigma, rho, kappa, v0) * (differential_A_sigma(u, tau, theta, sigma, rho, kappa) +
                                                                         v0 * differential_B_sigma(u, tau, sigma, rho, kappa))

In [42]:
def differential_wrt_sigma(lm, r, tau, theta, sigma, rho, kappa, v0):  
    m = lm + r*tau  #log-moneyness forward      
    integrand = (lambda u: 
        np.real(np.exp((1j*u + 0.5)*m) * differential_phi_sigma(u - 0.5j, tau, theta, sigma, rho, kappa, v0))/(u**2 + 0.25))

    integ, err = quad_vec(integrand, 0, np.inf)
    return (-1/np.pi) * integ   

## <Strong><font>3.8. Differential of p_hat wrt. r </font></Strong>

In [44]:
def differential_wrt_r(lm, r, tau, theta, sigma, rho, kappa, v0):
    diff_wrt_lm = differential_wrt_lm(lm, r, tau, theta, sigma, rho, kappa, v0)
    return tau * diff_wrt_lm

In [45]:
def test_differential_wrt_r():
    delta_r = 0.00000001
    r0 = 0.03
    p_hat_deltarplusr0 = norm_forw_put_price(lm=1., r=delta_r+r0, tau=5., theta=0.7,  sigma=0.7, rho=-0.6, kappa=0.15, v0=0.25)
    p_hat_r0 = norm_forw_put_price(lm=1., r=r0, tau=5., theta=0.7,  sigma=0.7, rho=-0.6, kappa=0.15, v0=0.25)

    differential_formula = differential_wrt_r(lm=1., r=0.03, tau=5., theta=0.7, sigma=0.7, rho=-0.6, kappa=0.15, v0=0.25)
    differential_finite_difference = (p_hat_deltarplusr0 - p_hat_r0)/delta_r

    np.testing.assert_almost_equal(differential_formula, differential_finite_difference, decimal=8, err_msg='', verbose=True)   

test_differential_wrt_r()    

# <strong> 4. Data generation </strong>

## <Strong><font> 4.1. Inputs generation </font></Strong> 

$\theta$, $\sigma$, $\kappa$, $\rho$ parameters, $v_0$, Log moneyness $lm=\ln(S_0/K)$, the time to maturity $\tau$ and $r$ will be sampled via Sobol Sampling technique. 

In [ ]:
def next_power_two(n: int) -> int:
    if n <= 1:
        return 1
    return 1 << (n - 1).bit_length()

n_samples = 10
# A sample consists of a row in the form (lm, r, tau, theta, sigma, rho, kappa, v0)
lower_bounds = [-1.5, -0.01, 1/52, 0.0, 0.1, -0.95, 0.05, 0.0]
upper_bounds = [1.5, 0.10, 3.0, 1.0, 1.5, 0.0, 5.0, 1.0]

sobol = qmc.Sobol(d=8, scramble=True, seed=42)
m = int(np.log2(next_power_two(n_samples)))
unit = sobol.random_base2(m=m)[:n_samples]
inputs_array = qmc.scale(unit, lower_bounds, upper_bounds)


In [67]:
def generate_inputs(d, n, lower_bounds, upper_bounds, feller=False):
    
    sobol = qmc.Sobol(d=d, scramble=True, seed=42)
    m = int(np.log2(next_power_two(n)))
    unit = sobol.random_base2(m=m)[:n]
    inputs_array = qmc.scale(unit, lower_bounds, upper_bounds)
    
    if feller:
        #Selecting samples that satisfy the Feller condition        
        inputs_array = inputs_array[np.where(2*inputs_array[:,6]*inputs_array[:,3] > (inputs_array[:,4])**2)]

    return inputs_array
    

## <Strong><font> 4.2. Labels and their differentials w.r.t inputs generation </font></Strong> 

In [ ]:
def generate_labels_difflabels(inputs_array):

    # array containing labels and its differentials
    labels_difflabels_array = np.zeros((inputs_array.shape[0],inputs_array.shape[1]+1))

    for row in range(inputs_array.shape[0]):

        lm = inputs_array[row][0] 
        r = inputs_array[row][1]
        tau = inputs_array[row][2]
        theta = inputs_array[row][3]
        sigma = inputs_array[row][4]
        rho = inputs_array[row][5]      
        kappa = inputs_array[row][6]
        v0 = inputs_array[row][7]    

        p_hat = norm_forw_put_price(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_lm = differential_wrt_lm(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_r = differential_wrt_r(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_tau = differential_wrt_tau(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_theta = differential_wrt_theta(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_sigma = differential_wrt_sigma(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_rho = differential_wrt_rho(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_kappa = differential_wrt_kappa(lm, r, tau, theta,sigma, rho, kappa, v0)
        diff_wrt_v0 = differential_wrt_v0(lm, r, tau, theta,sigma, rho, kappa, v0)

        labels_difflabels_array[row][0] = p_hat 
        labels_difflabels_array[row][1] = diff_wrt_lm
        labels_difflabels_array[row][2] = diff_wrt_r
        labels_difflabels_array[row][3] = diff_wrt_tau
        labels_difflabels_array[row][4] = diff_wrt_theta
        labels_difflabels_array[row][5] = diff_wrt_sigma
        labels_difflabels_array[row][6] = diff_wrt_rho
        labels_difflabels_array[row][7] = diff_wrt_kappa
        labels_difflabels_array[row][8] = diff_wrt_v0          

    return labels_difflabels_array


## <Strong><font> 4.3. Dataset generation </font></Strong> 

In [ ]:
def generate_data(d, n, lower_bounds, upper_bounds, feller, first_samples=100):

    inputs_array = generate_inputs(d, n, lower_bounds, upper_bounds, feller)

    labels_difflabels_array = generate_labels_difflabels(inputs_array)

    # Convert the inputs array to a dataframe
    df_inputs = pd.DataFrame(inputs_array, columns=['lm', 'r', 'tau', 'theta', 'sigma', 'rho', 'kappa', 'v0'])

    # Convert the array containing labels and its differentials to a dataframe
    df_labels_difflabels = pd.DataFrame(labels_difflabels_array, columns=["p_hat", "diff wrt lm", "diff wrt r", "diff wrt tau", "diff wrt theta",
                                                "diff wrt sigma", "diff wrt rho", "diff wrt kappa", "diff wrt v0"])

    # put the whole data into a dataframe
    df_dataset = pd.concat([df_inputs, df_labels_difflabels], axis=1) 
    df_dataset["S"] = 1.0
    df_dataset["K"] = df_dataset["S"] / np.exp(df_dataset["lm"]) 

        
    return df_dataset

### 4.3.1 Data generation with normalized forward prices and greeks
(Feller condition is breached)

In [ ]:
# A sample consists of a row in the form (lm, r, tau, theta, sigma, rho, kappa, v0)

df_nofeller = generate_data(d=8, n=5000,                                      
                                     lower_bounds = [-1.5, -0.01, 1/52, 0.0, 0.1, -0.95, 0.05, 0.0], 
                                     upper_bounds = [1.5, 0.10, 3.0, 1.0, 1.5, 0.0, 5.0, 1.0],
                                     feller=False)

# # Saving datasets into CSV files
df_nofeller.to_csv('../../data/bilokon/dataset_norm_forward_5K_nofeller.csv', sep=',', index=False)

### 4.3.2 New dataset with absolute discounted prices and greeks

In [83]:
df_norm_forward = pd.read_csv('../../data/bilokon/dataset_norm_forward_5K_nofeller.csv')

df_final = pd.DataFrame()

df_final["kappa"] = df_norm_forward["kappa"]  
df_final["theta"] = df_norm_forward["theta"]
df_final["sigma"] = df_norm_forward["sigma"]
df_final["rho"] = df_norm_forward["rho"]  
df_final["v0"] = df_norm_forward["v0"]
df_final["T"] = df_norm_forward["tau"]
df_final["r"] = df_norm_forward["r"]
df_final["lm"] = df_norm_forward["lm"]
df_final["S"] = df_norm_forward["S"]
df_final["K"] = df_norm_forward["K"]    


df_final["price"] = df_norm_forward["p_hat"]*df_norm_forward["K"]*np.exp(-df_norm_forward["r"]*df_norm_forward["tau"])
df_final["dtheta"] = df_norm_forward["diff wrt theta"]*df_norm_forward["K"]*np.exp(-df_norm_forward["r"]*df_norm_forward["tau"])
df_final["dv0"] = df_norm_forward["diff wrt v0"]* df_norm_forward["K"]*np.exp(-df_norm_forward["r"]*df_norm_forward["tau"])
df_final["dsigma"] = df_norm_forward["diff wrt sigma"]*df_norm_forward["K"]*np.exp(-df_norm_forward["r"]*df_norm_forward["tau"])
df_final["drho"] = df_norm_forward["diff wrt rho"]* df_norm_forward["K"]*np.exp(-df_norm_forward["r"]*df_norm_forward["tau"])
df_final["dkappa"] = df_norm_forward["diff wrt kappa"]* df_norm_forward["K"]*np.exp(-df_norm_forward["r"]*df_norm_forward["tau"])

df_final.to_csv('../../data/bilokon/dataset_abs_disc_5K_nofeller.csv', sep=',', index=False)


## <Strong><font> 4.4. Removing rows with unreal prices </font></Strong> 

### 4.4.1 Removing rows with negative prices

In [92]:

df_final = pd.read_csv('../../data/bilokon/dataset_abs_disc_5K_nofeller.csv')

# Identifiera negativa priser för loggning
neg_mask = df_final["price"] < 0
print(f"Antal rader med negativa priser som tas bort: {neg_mask.sum()} / {len(df_final)}")

# Behåll endast rader där priset är 0.0 eller högre
df_final = df_final[df_final["price"] >= 0].copy()

# Återställ index så att vi inte har luckor från de borttagna raderna
df_final.reset_index(drop=True, inplace=True)

print(f"Antal rader kvar: {len(df_final)}")
print(f"Minsta pris nu: {df_final['price'].min()}")

Antal rader med negativa priser som tas bort: 5 / 5000
Antal rader kvar: 4995
Minsta pris nu: 1.0412914510547062e-15


### 4.4.2 Removing rows with arbitrage prices

In [93]:
# Check if prices satisfy intrinsic value bound: p >= max(K*e^(-rT) - S, 0)

S = df_final["S"].values
K = df_final["K"].values
T = df_final["T"].values
r = df_final["r"].values

intrinsic = np.maximum(K * np.exp(-r * T) - S, 0.0)
prices = df_final["price"].values

#valid = (prices >= intrinsic - 1e-10).sum()
valid = (prices >= intrinsic).sum()
print(f"Priser som uppfyller p >= max(K*e^(-rT) - S, 0): {valid} / {len(prices)}")

Priser som uppfyller p >= max(K*e^(-rT) - S, 0): 4986 / 4995


In [94]:
# Parametrar
S = df_final["S"].to_numpy()
K = df_final["K"].to_numpy()
T = df_final["T"].to_numpy()
r = df_final["r"].to_numpy()
prices = df_final["price"].to_numpy()

# Diskonterade arbitragegränser för put (q = 0)
lower_disc = np.maximum(K * np.exp(-r * T) - S, 0.0)
upper_disc = K * np.exp(-r * T)

eps = 1e-12

# Alla brott mot bounds
any_lower_violation = prices < lower_disc
any_upper_violation = prices > upper_disc

# Små brott (inom eps)
tiny_lower_violation = (prices < lower_disc) & (prices >= lower_disc - eps)
tiny_upper_violation = (prices > upper_disc) & (prices <= upper_disc + eps)

# Riktiga brott (mer än eps)
real_lower_violation = prices < (lower_disc - eps)
real_upper_violation = prices > (upper_disc + eps)

print(f"Uppfyller lower bound: {(prices >= lower_disc).sum()} / {len(prices)}")
print(f"Uppfyller upper bound: {(prices <= upper_disc).sum()} / {len(prices)}")

print(f"Alla lower-brott: {any_lower_violation.sum()} / {len(prices)}")
print(f"Små lower-brott inom eps: {tiny_lower_violation.sum()} / {len(prices)}")
print(f"Riktiga lower-brott > eps: {real_lower_violation.sum()} / {len(prices)}")

print(f"Alla upper-brott: {any_upper_violation.sum()} / {len(prices)}")
print(f"Små upper-brott inom eps: {tiny_upper_violation.sum()} / {len(prices)}")
print(f"Riktiga upper-brott > eps: {real_upper_violation.sum()} / {len(prices)}")

# Radera rader med riktiga brott
real_violation_mask = real_lower_violation | real_upper_violation
deleted_count = real_violation_mask.sum()
df_final = df_final[~real_violation_mask].reset_index(drop=True)

print(f"Antal riktiga brott raderade: {deleted_count}")
print(f"Antal rader kvar: {len(df_final)}")

df_final.to_csv('../../data/bilokon/dataset_abs_disc_arbitrage_5K_nofeller.csv', sep=',', index=False)


Uppfyller lower bound: 4986 / 4995
Uppfyller upper bound: 4995 / 4995
Alla lower-brott: 9 / 4995
Små lower-brott inom eps: 3 / 4995
Riktiga lower-brott > eps: 6 / 4995
Alla upper-brott: 0 / 4995
Små upper-brott inom eps: 0 / 4995
Riktiga upper-brott > eps: 0 / 4995
Antal riktiga brott raderade: 6
Antal rader kvar: 4989


## <Strong><font> 4.5. Generating IV and IV-Greeks </font></Strong> 

### 4.5.1 Generating IV with Let's Be Rational

In [95]:
df_final = pd.read_csv('../../data/bilokon/dataset_abs_disc_arbitrage_5K_nofeller.csv')

n_samples = len(df_final)

lm  = df_final["lm"].to_numpy()                 # lm = ln(S/K)
T   = df_final["T"].to_numpy()
r   = df_final["r"].to_numpy()
S   = df_final["S"].to_numpy()                  # Normaliserat underliggande, dvs S = 1.0 
K   = df_final["K"].to_numpy() 
price = df_final["price"].to_numpy().copy()     # diskonterat och absolut
P_forward  = price * np.exp(r * T)              # odiskonterat och absolut put-pris då lets be rational förväntar sig den input formen        
q = 0.0
qflag = -1                                      # Put-flagga -1 i let´s be rational bibloteket

m = lm + (r - q) * T                            # m = ln(F/K)
F = np.exp(m) * K 

lower = np.maximum(K - F, 0.0)                  # Arbitragegränser. Görs i forward form då det är inputen i lbr
upper = K
eps = 1e-12

iv_unreal_mask = ~np.isfinite(P_forward) | (P_forward < lower) | (P_forward > upper)    # Inget reelt IV när pris är utanför detta. Sätter deras IV till NaN

iv = np.full(len(P_forward), np.nan, dtype=float)

print(f"Arbitrage IV's: {len(iv[iv_unreal_mask])} / {n_samples}")
print(f"Arbitrage IV's under intrisic: {(P_forward < lower).sum()} (+{(~np.isfinite(P_forward)).sum()} NaN) / {n_samples}")

# Kör Let’s Be Rational
for i, (P_i, F_i, K_i, T_i) in enumerate(zip(P_forward, F, K, T)):
    if iv_unreal_mask[i]:
        continue
    iv[i] = lbr.implied_volatility_from_a_transformed_rational_guess(P_i, F_i, K_i, T_i, qflag)

df_final["IV"] = iv                             # IV diskonteras/normaliseras ej utan är endast volatilitetsskala
print("Min IV:", np.nanmin(iv))
print("Max IV:", np.nanmax(iv))
print("Antal NaN IV:", np.sum(np.isnan(iv)))

#Tester för epsilon & masker:
gap = P_forward - lower

near_intrinsic = np.abs(gap) <= eps 

print("Antal nära intrinsic:", near_intrinsic.sum())
iv_near = iv[near_intrinsic]

print("NaN i near intrinsic:", np.sum(np.isnan(iv_near)))
print("Min IV:", np.nanmin(iv_near))
print("Max IV:", np.nanmax(iv_near))
print("Percentiler IV:", np.nanpercentile(iv_near, [0, 1, 5, 50, 95]))

low_idx = np.where(near_intrinsic)[0]

print("Antal low-mask:", len(low_idx))
print("Antal under lower inom low-mask:", np.sum(P_forward[low_idx] < lower[low_idx]))
print("Antal över/på lower inom low-mask:", np.sum(P_forward[low_idx] >= lower[low_idx]))

print("Gap i low-mask, min/max:")
print(np.min(gap[low_idx]), np.max(gap[low_idx]))

valid = np.isfinite(iv_near) & (iv_near > 0)

print("Median (valid IV):", np.median(iv_near[valid]))

Arbitrage IV's: 3 / 4989
Arbitrage IV's under intrisic: 3 (+0 NaN) / 4989
Min IV: 0.0
Max IV: 1.2439536969455
Antal NaN IV: 3
Antal nära intrinsic: 33
NaN i near intrinsic: 3
Min IV: 0.0
Max IV: 1.2439536969455
Percentiler IV: [0.         0.04904626 0.17336093 0.6216064  1.16142438]
Antal low-mask: 33
Antal under lower inom low-mask: 3
Antal över/på lower inom low-mask: 30
Gap i low-mask, min/max:
-3.5083047578154947e-13 7.491784970170556e-13
Median (valid IV): 0.623709756958777


### 4.5.2 Generating Vega

In [96]:
# Vega = F * phi(d1) * sqrt(T)

mask = (np.isfinite(iv) & (iv > 0) &         # Markerar rader där IV, F, T är ändliga och positiva (boolean mask)
        np.isfinite(F) & (F > 0.0) &             # Endast iv mask som gör skillnad för vårt dataset, 121 st iv=0
        np.isfinite(T) & (T > 0) &
        np.isfinite(K) & (K > 0))                  

d1 = np.full_like(iv, np.nan)                                                                      # d1[mask] = (np.log(F[mask]) + 0.5 * iv[mask]**2 * T[mask]) / (iv[mask] * np.sqrt(T[mask]))
d1[mask] = (np.log(F[mask]/K[mask]) + 0.5 * iv[mask]**2 * T[mask]) / (iv[mask] * np.sqrt(T[mask])) # Antar raden över K=1? Isf är detta riktiga formeln?

inv_sqrt_2pi = 1.0 / np.sqrt(2*np.pi)
phi_d1 = np.full_like(iv, np.nan)
phi_d1[mask] = inv_sqrt_2pi * np.exp(-0.5 * d1[mask]**2)

vega = np.full_like(iv, np.nan)
vega[mask] = np.exp(-r[mask] * T[mask]) * F[mask] * phi_d1[mask] * np.sqrt(T[mask])   # Diskonterat (som pris greeksen)

df_final["vega"] = vega

### 4.5.3 Generating IV-Greeks

In [97]:
PARAM_ORDER = ["kappa", "theta", "sigma", "rho", "v0", "T", "r", "lm"]

for k in PARAM_ORDER:
    if f"d{k}" not in df_final.columns:
        continue

    df_final[f'IVd{k}'] = df_final[f'd{k}'] / df_final['vega']

pd.set_option('display.max_columns', None)
print(df_final.head())
print(len(df_final))


      kappa     theta     sigma       rho        v0         T         r  \
0  3.503990  0.053457  0.517115 -0.152116  0.603962  2.422961  0.079580   
1  0.141656  0.995635  1.417571 -0.527489  0.349069  1.392509  0.004078   
2  4.713475  0.436169  0.957227 -0.833994  0.207538  2.248123  0.050174   
3  1.426669  0.612892  0.277241 -0.261222  0.964370  0.076599  0.043582   
4  3.837604  0.657712  0.253053 -0.013875  0.063382  0.829567  0.071719   

         lm    S         K     price        dtheta           dv0  \
0 -0.206912  1.0  1.229874  0.216646  7.787843e-01  1.033986e-01   
1  1.326168  1.0  0.265493  0.015627  3.291129e-03  3.800736e-02   
2  0.314995  1.0  0.729793  0.156492  2.572036e-01  2.669359e-02   
3 -1.434252  1.0  4.196503  3.182517  2.430144e-09  4.315393e-08   
4 -1.033181  1.0  2.809989  1.673816  8.583034e-02  3.699312e-02   

         dsigma          drho        dkappa        IV          vega  IVdkappa  \
0 -9.207347e-03  8.068711e-03 -1.493016e-02  0.338840  6.03

## <Strong><font> 4.6. Saving full dataset </font></Strong> 

In [98]:
from pathlib import Path

out_dir = Path("../../data/bilokon")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"full_bilokon_dataset_5K_nofeller.csv"
df_final.to_csv(out_path, index=False)

print(f"Sparade {len(df_final)} rader till {out_path.resolve()}")
print(f"Kolumner: {list(df_final.columns)}")

Sparade 4989 rader till /Users/jacobhansen/Desktop/Kanarb/Kandidatarbete/data/bilokon/full_bilokon_dataset_5K_nofeller.csv
Kolumner: ['kappa', 'theta', 'sigma', 'rho', 'v0', 'T', 'r', 'lm', 'S', 'K', 'price', 'dtheta', 'dv0', 'dsigma', 'drho', 'dkappa', 'IV', 'vega', 'IVdkappa', 'IVdtheta', 'IVdsigma', 'IVdrho', 'IVdv0']
